# AirShift — Deterioration Event Definition

This notebook defines what constitutes a future air quality deterioration event for the AirShift early warning system.

The goal is to establish a clear and measurable definition of deterioration that can later be used to create target labels for machine learning.

The definition focuses on:

* The pollutant used to represent deterioration
* The magnitude of the deterioration
* The future time window in which deterioration is evaluated

No target labels are created in this notebook. The defined event will be used in the following labeling stage.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path("../data/processed")

feature_file = DATA_DIR / "airshift_feature_engineered.csv"

df = pd.read_csv(feature_file, parse_dates=["datetime"])

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

Dataset shape: (420768, 99)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 23:00:00


## 1. Define the Deterioration Event

AirShift focuses on detecting significant future deterioration in air quality before it occurs.

For this project, **PM2.5** is used as the primary pollutant for defining a deterioration event because it is an important indicator of particulate air pollution.

A deterioration event is defined as a **30% or greater increase in PM2.5 within the following 6 hours** compared with the PM2.5 level at the prediction time.

The event definition is therefore based on:

* **Target pollutant:** PM2.5
* **Deterioration threshold:** 30% increase
* **Future horizon:** 6 hours

The future deterioration value will be used only to define the target in the labeling stage. It will not be used as an input feature for model prediction.


## 1. Define the Deterioration Event

AirShift focuses on detecting significant future deterioration in air quality before it occurs.

For this project, **PM2.5** is used as the primary pollutant for defining a deterioration event because it is an important indicator of particulate air pollution.

A deterioration event is defined as a **30% or greater increase in PM2.5 within the following 6 hours** compared with the PM2.5 level at the prediction time.

The event definition is therefore based on:

* **Target pollutant:** PM2.5
* **Deterioration threshold:** 30% increase
* **Future horizon:** 6 hours

The future deterioration value will be used only to define the target in the labeling stage. It will not be used as an input feature for model prediction.


In [3]:
TARGET_POLLUTANT = "PM2.5"

DETERIORATION_THRESHOLD = 0.30

FUTURE_HORIZON_HOURS = 6

print("Target pollutant:", TARGET_POLLUTANT)
print("Deterioration threshold:", f"{DETERIORATION_THRESHOLD * 100:.0f}%")
print("Future horizon:", f"{FUTURE_HORIZON_HOURS} hours")

Target pollutant: PM2.5
Deterioration threshold: 30%
Future horizon: 6 hours


## 2. Calculate Future PM2.5 Values

To define a future deterioration event, the PM2.5 concentration at the prediction time is compared with its value 6 hours later.

The future PM2.5 value is calculated separately for each monitoring station to preserve the temporal structure of the data.

These future values are used only to evaluate the deterioration event definition and will later be converted into target labels in the labeling stage.


In [4]:
df = df.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)

df["PM2.5_future_6h"] = (
    df.groupby("station")["PM2.5"]
      .shift(-FUTURE_HORIZON_HOURS)
)

print("Future PM2.5 values calculated successfully.")

df[
    [
        "station",
        "datetime",
        "PM2.5",
        "PM2.5_future_6h"
    ]
].head(10)

Future PM2.5 values calculated successfully.


,station,datetime,PM2.5,PM2.5_future_6h
0,Aotizhongxin,2013-03-01 00:00:00,4.0,3.0
1,Aotizhongxin,2013-03-01 01:00:00,8.0,3.0
2,Aotizhongxin,2013-03-01 02:00:00,7.0,3.0
3,Aotizhongxin,2013-03-01 03:00:00,6.0,3.0
4,Aotizhongxin,2013-03-01 04:00:00,3.0,3.0
5,Aotizhongxin,2013-03-01 05:00:00,5.0,3.0
6,Aotizhongxin,2013-03-01 06:00:00,3.0,3.0
7,Aotizhongxin,2013-03-01 07:00:00,3.0,3.0
8,Aotizhongxin,2013-03-01 08:00:00,3.0,6.0
9,Aotizhongxin,2013-03-01 09:00:00,3.0,8.0


### Future PM2.5 Values Findings

The future PM2.5 values were calculated successfully for each monitoring station using a 6-hour horizon.

The `PM2.5_future_6h` feature represents the PM2.5 concentration measured six hours after each observation at the same monitoring station.

This future value will be used in the next step to calculate the relative change in PM2.5 and determine whether the 30% deterioration threshold is reached.

No target labels are created at this stage.
